# 04 - Unified benchmark: Qwen2.5-0.5B-Instruct and Gemma-2-2b-it (base vs DPO)

Evaluates the **base** and **DPO-fine-tuned** variants of both model families on:
* standard benchmarks via `lm-eval`: ARC-Challenge, HellaSwag, Winogrande, TruthfulQA-MC2, MMLU, IFEval
* **preference accuracy** (does the model give higher log-probability to `chosen` than to `rejected`?) on the held-out test split, both raw (sum of log-probs) and length-normalised (mean log-prob per token)

**How to run**
1. Run the setup cells once.
2. For each of the 4 combinations (`qwen/base`, `qwen/dpo`, `gemma/base`, `gemma/dpo`): set `MODEL_FAMILY` and `MODEL_CHOICE` in the config cell, then run *load test set -> load model -> preference accuracy -> lm-eval*.
3. Run the cleanup cell between combinations to free VRAM.
4. After all four runs, run the final comparison cells.

> The preference test set and the trained adapters are not distributed with this repo;
> point the paths in the config cell to your own files.

**Caveat:** with `LIMIT = 250`, every lm-eval task is scored on at most 250 examples
(standard error of roughly +-3 points on a ~50% accuracy), so differences of a few points
between base and DPO models are within noise. Raise `LIMIT` (or set it to `None`) for stable estimates.

> 🔒 **Data availability.** The preference datasets used in this work (and the AutoBench export they are derived from) were shared privately by the AutoBench authors and are **not included in this repository**. All paths in this notebook are placeholders (`<your-...>`): point them at your own data with the same schema (`prompt`, `chosen`, `rejected`). Cell outputs have been cleared so that no private data is exposed.

## Setup

In [ ]:
import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "lm-eval[ifeval]",
    "unsloth[kaggle-new]",
    "--quiet", "--upgrade"
])

import lm_eval, unsloth
print(f"lm-eval {lm_eval.__version__}")
print(f"unsloth {unsloth.__version__}")

In [ ]:
# Environment: variables, cache, HuggingFace login, GPU info.
# The login is needed because Gemma-2-2b-it is a gated model
# (Qwen2.5-0.5B-Instruct is not, but logging in is harmless).
import os

os.environ["CUDA_VISIBLE_DEVICES"]    = "0"
os.environ["TOKENIZERS_PARALLELISM"]  = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

BASE_DIR  = "/kaggle/working"
CACHE_DIR = os.path.join(BASE_DIR, "hf_cache")
os.environ["HF_HOME"]            = CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = os.path.join(CACHE_DIR, "transformers")
os.environ["HF_DATASETS_CACHE"]  = os.path.join(CACHE_DIR, "datasets")
os.makedirs(CACHE_DIR, exist_ok=True)

import torch
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secret = UserSecretsClient()
hf_token = secret.get_secret("HF_TOKEN")
login(token=hf_token)
print("Authenticated on Hugging Face")

## Configuration and data (repeat for every family/variant combination)

In [ ]:
# The notebook runs four times: {qwen, gemma} x {base, dpo}.
#   1. MODEL_FAMILY="qwen",  MODEL_CHOICE="base"
#   2. MODEL_FAMILY="qwen",  MODEL_CHOICE="dpo"
#   3. MODEL_FAMILY="gemma", MODEL_CHOICE="base"
#   4. MODEL_FAMILY="gemma", MODEL_CHOICE="dpo"
#   5. Finally run the comparison cells once (they read all saved result files).

MODEL_FAMILY = "qwen"   # "qwen" | "gemma"
MODEL_CHOICE = "base"   # "base" | "dpo"

# -- Per-family configuration ------------------------------------
FAMILY_CONFIG = {
    "qwen": {
        "base_model_id": "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit",
        "adapter_path":  "/kaggle/input/<your-qwen-adapter>",  # trained adapters are not distributed   # folder with adapter_config.json
        "official_baseline": {   # published numbers, for reference only
            "arc_challenge": 0.339, "hellaswag": 0.409, "winogrande": 0.553,
            "truthfulqa_mc2": 0.419, "mmlu": 0.474, "ifeval": 0.229,
        },
    },
    "gemma": {
        "base_model_id": "unsloth/gemma-2-2b-it-bnb-4bit",
        "adapter_path":  "/kaggle/input/<your-gemma-adapter>",  # folder with adapter_config.json
        "official_baseline": {
            "arc_challenge": 0.524, "hellaswag": 0.742, "winogrande": 0.715,
            "truthfulqa_mc2": 0.450, "mmlu": 0.528,
            "ifeval": None,   # not reported for this model
        },
    },
}

cfg = FAMILY_CONFIG[MODEL_FAMILY]
BASE_MODEL_ID     = cfg["base_model_id"]
ADAPTER_PATH      = cfg["adapter_path"]
OFFICIAL_BASELINE = cfg["official_baseline"]

# -- Standard benchmarks (lm-eval) -------------------------------
TASKS = ["arc_challenge", "hellaswag", "winogrande", "truthfulqa_mc2", "mmlu", "ifeval"]

METRIC_MAP = {
    "arc_challenge":  "acc_norm,none",
    "hellaswag":      "acc_norm,none",
    "winogrande":     "acc,none",
    "truthfulqa_mc2": "acc,none",
    "mmlu":           "acc,none",
    "ifeval":         "prompt_level_strict_acc,none",
}

# -- Evaluation parameters ---------------------------------------
BATCH_SIZE     = 4     # lower to 2 on OOM (more likely with Gemma-2-2b)
MAX_NEW_TOKENS = 256
NUM_FEWSHOT    = 0     # zero-shot, consistent with the reference baselines above
LIMIT          = 250   # examples per task; raise (or None) for more stable estimates

# -- Output ------------------------------------------------------
RESULTS_DIR = os.path.join("/kaggle/working", "results")
os.makedirs(RESULTS_DIR, exist_ok=True)
RUN_ID = f"{MODEL_FAMILY}_{MODEL_CHOICE}"     # e.g. "qwen_base", "gemma_dpo"

# -- Preference test set (chosen vs rejected) --------------------
# The same test set is used for both families so the comparison is fair.
# NOTE: private data - the preference test split is not distributed with this repo.
TEST_SET_PATH   = "/kaggle/input/<your-dpo-dataset>/dpo_best_worst_test.parquet"
PREF_EVAL_LIMIT = None   # None = evaluate every triple

print(f"Family        : {MODEL_FAMILY.upper()}")
print(f"Variant       : {MODEL_CHOICE.upper()}")
print(f"Base model    : {BASE_MODEL_ID}")
print(f"Adapter (dpo) : {ADAPTER_PATH}")
print(f"Benchmarks    : {TASKS}")
print(f"Pref test set : {TEST_SET_PATH}")
print(f"Output        : {RESULTS_DIR}")
print(f"RUN_ID        : {RUN_ID}")

In [ ]:
# Load the preference test set (chosen vs rejected)
from datasets import load_dataset

ext = TEST_SET_PATH.rsplit(".", 1)[-1].lower()
fmt = {"json": "json", "jsonl": "json", "csv": "csv", "parquet": "parquet"}.get(ext, "json")
ds_test = load_dataset(fmt, data_files=TEST_SET_PATH, split="train")
ds_test = ds_test.select_columns(["prompt", "chosen", "rejected"])

if PREF_EVAL_LIMIT is not None:
    ds_test = ds_test.select(range(min(PREF_EVAL_LIMIT, len(ds_test))))

print(f"Test set loaded: {len(ds_test):,} triples")

## Load the model

Run **exactly one** of the two loader cells below (Option A is the default).

### Option A - Unsloth loader (default)

In [ ]:
import torch
from unsloth import FastLanguageModel
from peft import PeftModel

torch.cuda.empty_cache()
print(f"Loading model '{RUN_ID}'...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL_ID,
    max_seq_length = 2048,
    dtype          = None,
    load_in_4bit   = True,
    cache_dir      = CACHE_DIR,
)

if MODEL_CHOICE == "dpo":
    if not os.path.exists(os.path.join(ADAPTER_PATH, "adapter_config.json")):
        raise FileNotFoundError(
            f"adapter_config.json not found in: {ADAPTER_PATH}\n"
            f"Check that the adapter folder is mounted correctly.")
    model = PeftModel.from_pretrained(model, ADAPTER_PATH)
    # model = model.merge_and_unload()   # optional: merge the adapter for faster inference
    print(f"   DPO adapter ({MODEL_FAMILY.upper()}) loaded")
elif MODEL_CHOICE == "base":
    print("   No adapter loaded - plain base model")
else:
    raise ValueError("MODEL_CHOICE must be 'base' or 'dpo'")

FastLanguageModel.for_inference(model)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Model ready - VRAM used: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

### Option B - Transformers + PEFT loader (adapter merged into the weights)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    attn_implementation="eager",   # essential for Gemma 2: the logit soft-cap is only correct here
)

if MODEL_CHOICE == "dpo":
    if not os.path.exists(os.path.join(ADAPTER_PATH, "adapter_config.json")):
        raise FileNotFoundError(f"adapter_config.json not found in: {ADAPTER_PATH}")
    model = PeftModel.from_pretrained(model, ADAPTER_PATH)
    model = model.merge_and_unload()   # removes any ambiguity about which adapter is active
    print(f"DPO adapter ({MODEL_FAMILY.upper()}) loaded and merged")
else:
    print("No adapter loaded - plain base model")

model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Model ready - VRAM used: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

## Preference accuracy (chosen vs rejected)

For each test triple we compute the log-probability the model assigns to the `chosen` and to the
`rejected` completion given the prompt; a prediction is correct when `logp(chosen) > logp(rejected)`.

Two variants are reported:
* **raw** - total log-probability of the completion (penalises long answers),
* **normalised** - mean log-probability per token (reduces the length bias; see thesis Section 6.4).

In [ ]:
import json
import torch


@torch.no_grad()
def compute_seq_logp(model, tokenizer, prompt, completion, max_length=2048, normalize=True):
    """Log-probability of `completion` conditioned on `prompt`.

    normalize=True  -> mean log-prob per token (length-normalised)
    normalize=False -> total log-prob of the sequence (biased by length)
    """
    prompt_msgs     = [{"role": "user", "content": prompt}]
    completion_msgs = [{"role": "assistant", "content": completion}]
    prompt_ids = tokenizer.apply_chat_template(
        prompt_msgs, add_generation_prompt=True, return_tensors="pt")
    full_ids = tokenizer.apply_chat_template(
        prompt_msgs + completion_msgs, add_generation_prompt=False, return_tensors="pt")
    prompt_ids = prompt_ids.to(model.device)
    full_ids   = full_ids.to(model.device)
    if full_ids.shape[1] > max_length:
        full_ids = full_ids[:, :max_length]

    # Safety check on the prompt/completion tokenisation boundary
    prompt_len = prompt_ids.shape[1]
    match_len = min(prompt_len, full_ids.shape[1])
    mismatch = (prompt_ids[0, :match_len] != full_ids[0, :match_len]).nonzero()
    if len(mismatch) > 0:
        prompt_len = mismatch[0].item()   # the real boundary is earlier than assumed

    if full_ids.shape[1] <= prompt_len:
        return float("-inf")

    labels = full_ids.clone()
    labels[:, :prompt_len] = -100          # score only the completion tokens
    outputs = model(input_ids=full_ids, labels=labels)
    nll = outputs.loss.item()
    n_completion_tokens = (labels != -100).sum().item()
    if n_completion_tokens == 0:
        return float("-inf")

    avg_logp = -nll
    sum_logp = -nll * n_completion_tokens
    return avg_logp if normalize else sum_logp


def extract_user_prompt(messages):
    for msg in reversed(messages):
        if msg["role"] == "user":
            return msg["content"]
    return ""


def extract_assistant_answer(messages):
    for msg in reversed(messages):
        if msg["role"] == "assistant":
            return msg["content"]
    return ""


def evaluate_preference_accuracy(model, tokenizer, dataset, normalize=True):
    model.eval()
    correct, total = 0, 0
    for i, ex in enumerate(dataset):
        if (i + 1) % 50 == 0:
            print(f"  -> evaluating sample {i + 1}/{len(dataset)}")
        prompt   = extract_user_prompt(ex["prompt"])
        chosen   = extract_assistant_answer(ex["chosen"])
        rejected = extract_assistant_answer(ex["rejected"])
        logp_chosen   = compute_seq_logp(model, tokenizer, prompt, chosen, normalize=normalize)
        logp_rejected = compute_seq_logp(model, tokenizer, prompt, rejected, normalize=normalize)
        if logp_chosen != float("-inf") and logp_rejected != float("-inf"):
            correct += int(logp_chosen > logp_rejected)
            total += 1
    return correct / total if total > 0 else 0.0


print(f"Computing preference accuracy for: {RUN_ID.upper()}")

print("\n--- RAW version (sum of log-probs, length-biased) ---")
pref_accuracy_sum = evaluate_preference_accuracy(model, tokenizer, ds_test, normalize=False)
print(f"Preference accuracy (raw): {pref_accuracy_sum:.2%}")

print("\n--- NORMALISED version (mean log-prob per token) ---")
pref_accuracy_norm = evaluate_preference_accuracy(model, tokenizer, ds_test, normalize=True)
print(f"Preference accuracy (normalised): {pref_accuracy_norm:.2%}")

pref_output_file = os.path.join(RESULTS_DIR, f"results_preference_{RUN_ID}.json")
with open(pref_output_file, "w") as f:
    json.dump({
        "preference_accuracy_sum": pref_accuracy_sum,
        "preference_accuracy_avg_normalized": pref_accuracy_norm,
    }, f, indent=2)
print(f"\nSaved to: {pref_output_file}")

## Standard benchmarks (lm-eval)

In [ ]:
import json
import torch
from lm_eval import evaluator
from lm_eval.models.huggingface import HFLM

torch.cuda.empty_cache()

lm_model = HFLM(
    pretrained = model,
    tokenizer  = tokenizer,
    batch_size = BATCH_SIZE,
    max_length = 2048,
    dtype      = "float16",
)

print(f"Starting benchmarks for: {RUN_ID.upper()}")
print(f"   Tasks    : {TASKS}")
print(f"   Few-shot : {NUM_FEWSHOT}-shot\n")

results = evaluator.simple_evaluate(
    model       = lm_model,
    tasks       = TASKS,
    num_fewshot = NUM_FEWSHOT,
    batch_size  = BATCH_SIZE,
    device      = "cuda",
    log_samples = False,
    limit       = LIMIT,
)

output_file = os.path.join(RESULTS_DIR, f"results_{RUN_ID}.json")
with open(output_file, "w") as f:
    json.dump(results["results"], f, indent=2)
print(f"\nResults saved to: {output_file}")

print(f"\n{'=' * 55}")
print(f"  RESULTS - {RUN_ID.upper()}")
print(f"{'=' * 55}")
for task, metric_key in METRIC_MAP.items():
    if task in results["results"]:
        val = results["results"][task].get(metric_key, "N/A")
        if isinstance(val, float):
            print(f"  {task:<20} {metric_key:<35} {val * 100:.2f}%")
        else:
            print(f"  {task:<20} {metric_key:<35} {val}")
print(f"{'=' * 55}")

## Cleanup - run before switching to the next combination

In [ ]:
import gc

del model
del tokenizer
if "lm_model" in dir():
    del lm_model

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print(f"Free GPU memory after cleanup: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")
print("Ready for the next run: go back to the configuration cell and change MODEL_FAMILY / MODEL_CHOICE.")

## Final comparison (run only after completing all 4 combinations)

In [ ]:
import json, os
import pandas as pd

RUN_IDS = ["qwen_base", "qwen_dpo", "gemma_base", "gemma_dpo"]

METRIC_MAP_DISPLAY = {
    "arc_challenge":  "acc_norm,none",
    "hellaswag":      "acc_norm,none",
    "winogrande":     "acc,none",
    "truthfulqa_mc2": "acc,none",
    "mmlu":           "acc,none",
    "ifeval":         "prompt_level_strict_acc,none",
}

rows, missing_runs = [], []

# Folder containing the results_*.json files of all four runs
# (copy them into one place if you ran the combinations in separate sessions)
RESULTS_DIR = "/kaggle/working/results"

for run_id in RUN_IDS:
    path = os.path.join(RESULTS_DIR, f"results_{run_id}.json")
    if not os.path.exists(path):
        missing_runs.append(run_id)
        continue
    with open(path) as f:
        res = json.load(f)

    family, variant = run_id.split("_")
    row = {"family": family.upper(), "variant": variant.upper(), "run_id": run_id}

    for task, metric_key in METRIC_MAP_DISPLAY.items():
        if task not in res:
            row[task] = None
            continue
        val = (res[task].get(metric_key)
               or res[task].get(metric_key.split(",")[0])
               or res[task].get("acc,none")
               or res[task].get("acc_norm,none"))
        row[task] = round(val * 100, 2) if isinstance(val, float) else None

    # Preference accuracy: raw (sum of log-probs) and length-normalised
    pref_path = os.path.join(RESULTS_DIR, f"results_preference_{run_id}.json")
    if os.path.exists(pref_path):
        with open(pref_path) as f:
            pref = json.load(f)
        row["pref_acc_raw"]  = round(pref["preference_accuracy_sum"] * 100, 2)
        row["pref_acc_norm"] = round(pref["preference_accuracy_avg_normalized"] * 100, 2)
    else:
        row["pref_acc_raw"] = row["pref_acc_norm"] = None

    rows.append(row)

if missing_runs:
    print(f"WARNING: runs with no result file: {missing_runs}")
    print("         Complete those runs before interpreting the comparison below.\n")

METRIC_COLS = list(METRIC_MAP_DISPLAY.keys()) + ["pref_acc_raw", "pref_acc_norm"]

if not rows:
    print("No results found. Run at least one combination before this cell.")
else:
    df_results = pd.DataFrame(rows)

    # Delta DPO - BASE, per family
    delta_rows = []
    for family in ["QWEN", "GEMMA"]:
        base = df_results[(df_results["family"] == family) & (df_results["variant"] == "BASE")]
        dpo  = df_results[(df_results["family"] == family) & (df_results["variant"] == "DPO")]
        if base.empty or dpo.empty:
            continue
        base, dpo = base.iloc[0], dpo.iloc[0]
        delta = {"family": family, "variant": "DELTA DPO-BASE", "run_id": f"{family.lower()}_delta"}
        for col in METRIC_COLS:
            b, d = base.get(col), dpo.get(col)
            delta[col] = round(d - b, 2) if (b is not None and d is not None) else None
        delta_rows.append(delta)

    df_display = pd.concat([df_results, pd.DataFrame(delta_rows)], ignore_index=True)

    print("=" * 100)
    print("  FULL COMPARISON - BASE vs DPO, QWEN vs GEMMA")
    print("=" * 100)
    print(df_display.drop(columns=["run_id"]).to_string(index=False))
    print("\nPositive delta = the DPO model improved over its own base model.\n")

## Export the comparison as JSON (handy for plots or the README)

In [ ]:
chart_payload = {"tasks": METRIC_COLS, "series": {}}

for _, row in df_display.iterrows():
    if row["variant"] in ("BASE", "DPO"):
        key = f"{row['family']}_{row['variant']}"
        chart_payload["series"][key] = {t: row.get(t) for t in chart_payload["tasks"]}

print(json.dumps(chart_payload, indent=2))

chart_payload_path = os.path.join(RESULTS_DIR, "chart_payload.json")
with open(chart_payload_path, "w") as f:
    json.dump(chart_payload, f, indent=2)
print(f"\nSaved to: {chart_payload_path}")